# 04 — Ablação do gating (GmNet-S3 do zero)

Mede a contribuição do mecanismo de gating do GmNet isolando-o: duas variantes do GmNet-S3
treinadas **do zero** (sem ImageNet) no `cerrado_df_v1`, idênticas exceto pela operação do bloco:

| variante | config | bloco |
|---|---|---|
| com gating | `configs/experiments/gmnet_s3_scratch.yaml` | `ReLU6(x) * x` (original) |
| sem gating | `configs/experiments/gmnet_s3_nogate_scratch.yaml` | `ReLU6(x)` |

O gating não tem parâmetros: com a mesma semente, as duas variantes partem dos **mesmos pesos
iniciais** e veem os dados na mesma ordem.

**Piloto**: 1 semente (2026), 100 épocas cada (~1 h por variante numa T4). O número de sementes
é decidido depois (ver `docs/decisoes.md`). A receita do zero (LR 1e-3, 100 épocas, warmup 10)
é provisória.

**Antes**: rode o `00_setup_colab.ipynb` no mesmo ambiente. Use um ambiente com **GPU**.
Se o Colab desconectar, rode de novo: execuções concluídas são puladas e a interrompida retoma
do `last.pt`. Resultados para o artigo: só no modo `git`, sem alterações não commitadas.

In [ ]:
CONFIGS = {
    "com_gating": "configs/experiments/gmnet_s3_scratch.yaml",
    "sem_gating": "configs/experiments/gmnet_s3_nogate_scratch.yaml",
}
RUN_TESTS = True  # roda a suíte de testes (incluindo os de PyTorch) antes de treinar

## 1. Ambiente

In [ ]:
import importlib
import sys
from pathlib import Path

import torch
import yaml

REPO_DIR = Path("/content/ProjetoFinalRNAP_UnB2026")
assert REPO_DIR.exists(), "Rode antes o 00_setup_colab.ipynb neste ambiente."

# O kernel em execução não lê o .pth da instalação editável (ver notebook 00).
SRC_DIR = str(REPO_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
importlib.invalidate_caches()

from cerrado_flora.utils import run_metadata

paths = yaml.safe_load((REPO_DIR / "configs/paths.yaml").read_text())
cfgs = {name: yaml.safe_load((REPO_DIR / path).read_text()) for name, path in CONFIGS.items()}
assert all(c["model"]["pretrained"] is None for c in cfgs.values()), "a ablação é do zero"
datasets = {c["dataset"]["name"] for c in cfgs.values()}
assert len(datasets) == 1, "as duas variantes devem usar o mesmo dataset"

drive_root = Path(paths["drive_root"])
assert drive_root.exists(), "Google Drive não montado (seção 1 do notebook 00)."
images_root = Path(paths["local_cache"]) / datasets.pop()
assert (images_root / "images").exists(), "Dataset ausente: rode a seção 5 do notebook 00."
manifest = REPO_DIR / cfgs["com_gating"]["dataset"]["manifest"]

meta = run_metadata.collect(REPO_DIR)
print("commit:", meta["git_commit"], "| alterações não commitadas:", meta["git_dirty"])
if meta["git_commit"] is None or meta["git_dirty"]:
    print("AVISO: sem commit limpo; estes resultados não devem entrar no artigo (modo git).")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA")
assert torch.cuda.is_available(), "Selecione um ambiente com GPU."

## 2. Testes

In [ ]:
import subprocess

if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
    # -rfEs: resumo com falhas (f), erros (E) e testes pulados (s);
    # --tb=short: traceback curto de cada falha. Saída completa, sem cortes.
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-rfEs", "--tb=short", "tests"],
                       check=False, cwd=REPO_DIR, capture_output=True, text=True)
    print(r.stdout)
    if r.stderr:
        print(r.stderr)
    assert r.returncode == 0, "testes falharam (ver 'short test summary info' acima)"

## 3. Treino e avaliação

As variantes se alternam por semente (com e sem gating da semente 2026, depois da próxima...),
para que uma interrupção não deixe uma variante com mais sementes que a outra.

In [ ]:
from cerrado_flora.training.train import run_experiment

seeds = cfgs["com_gating"]["seeds"]
assert seeds == cfgs["sem_gating"]["seeds"]
results = {v: {} for v in cfgs}
for seed in seeds:
    for variant, cfg in cfgs.items():
        exp = cfg["name"]
        print(f"\n=== {variant} ({exp}), semente {seed}")
        results[variant][seed] = run_experiment(
            cfg, seed, manifest, images_root, None,
            ckpt_dir=drive_root / paths["drive_subdirs"]["checkpoints"] / exp / f"seed_{seed}",
            results_dir=drive_root / paths["drive_subdirs"]["results"] / exp / f"seed_{seed}",
        )

## 4. Resultados

In [ ]:
import pandas as pd

METRICS = ["top1", "macro_f1", "balanced_accuracy"]
rows = []
for variant, by_seed in results.items():
    for seed, r in by_seed.items():
        for split in ("val", "test"):
            for level in ("photo", "observation"):
                rows.append({"variante": variant, "seed": seed, "divisao": split, "nivel": level,
                             **{k: r[split][level][k] for k in METRICS},
                             "melhor_epoca": r["best_epoch"]})
df = pd.DataFrame(rows)
df.round(4)

Diferença pareada por semente (com gating − sem gating). Positivo = o gating ajudou.

In [ ]:
wide = df.pivot_table(index=["divisao", "nivel", "seed"], columns="variante", values=METRICS)
diff = pd.concat({m: wide[(m, "com_gating")] - wide[(m, "sem_gating")] for m in METRICS}, axis=1)
print(f"sementes: {sorted(results['com_gating'])}")
display(diff.round(4))
if diff.index.get_level_values("seed").nunique() > 1:
    display(diff.groupby(["divisao", "nivel"]).agg(["mean", "std"]).round(4))

In [ ]:
import matplotlib.pyplot as plt

seed = seeds[0]
colors = {"com_gating": "#2a78d6", "sem_gating": "#eb6834"}  # paleta categórica validada
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
for variant, cfg in cfgs.items():
    res_dir = drive_root / paths["drive_subdirs"]["results"] / cfg["name"] / f"seed_{seed}"
    h = pd.read_csv(res_dir / "history.csv")
    label = variant.replace("_", " ")
    a1.plot(h["epoch"], h["train_loss"], color=colors[variant], lw=2, label=f"{label}, treino")
    a1.plot(h["epoch"], h["val_loss"], color=colors[variant], lw=2, ls="--", label=f"{label}, validação")
    a2.plot(h["epoch"], h["val_macro_f1"], color=colors[variant], lw=2, label=label)
    a2.axvline(results[variant][seed]["best_epoch"], color=colors[variant], lw=1, ls=":")
    print(f"{variant}: tempo médio por época {h['epoch_time_s'].mean():.0f}s")
a1.set_title("Perda (treino com label smoothing)", loc="left")
a2.set_title(f"Macro-F1 por foto na validação (semente {seed})", loc="left")
for a in (a1, a2):
    a.set_xlabel("época (a partir de 0)")
    a.grid(color="#e5e4e0", lw=0.6)
    a.set_axisbelow(True)
    for side in ("top", "right"):
        a.spines[side].set_visible(False)
    a.legend(frameon=False)
plt.tight_layout()
plt.show()

F1 por classe no teste (por foto), primeira semente, ordenado pela diferença com − sem gating:

In [ ]:
species = pd.read_csv(manifest.parent / "species.csv").dropna(subset=["class_index"])
names = species.set_index(species["class_index"].astype(int))["scientific_name"]
f1 = pd.DataFrame({
    variant: pd.read_csv(drive_root / paths["drive_subdirs"]["results"] / cfg["name"] / f"seed_{seed}"
                         / "per_class_test.csv", index_col="class_index")["f1"]
    for variant, cfg in cfgs.items()
})
f1.insert(0, "especie", names)
f1["diferenca"] = f1["com_gating"] - f1["sem_gating"]
f1.sort_values("diferenca").round(3)